# Fine-tune a Hebrew NLI model on clean HebNLI  ·  Person B

`nli_rerank.py` currently defaults to `oriel9p/AlephBERT-FT-HebNLI-LCHAIM`, which was
fine-tuned on all of HebNLI — including the rows the probe was mined from. It has
already seen our (target, negation) pairs labelled `contradiction`, so its probe
scores are partly recall rather than judgement. This notebook builds the replacement.

Runtime → Change runtime type → **T4 GPU** before section 5. Sections 1–4 are CPU-only.

Order matters: the offline checks in section 1 cost seconds and catch the mistakes
that would otherwise surface an hour into training.

## 1. Clone and check

The repo is private. Create a classic PAT with `repo` scope and store it as the Colab
secret `GH_TOKEN`, so it never lands in the saved notebook or in git.

In [ ]:
OWNER, REPO, BRANCH = 'ItayBoros', 'hebrew-negation-embeddings', 'main'

import os, subprocess
from google.colab import userdata

gh_token = userdata.get('GH_TOKEN').strip()
url = f'https://{gh_token}@github.com/{OWNER}/{REPO}.git'

if not os.path.exists(REPO):
    subprocess.run(['git','clone','-q','--branch',BRANCH,url,REPO], check=True)
os.chdir(f'/content/{REPO}')
subprocess.run(['git','pull','-q','origin',BRANCH], check=True)

# drop the token from the stored remote so it is not left on disk
subprocess.run(['git','remote','set-url','origin',
                f'https://github.com/{OWNER}/{REPO}.git'], check=True)

!pip install -q -r requirements.txt
!git log --oneline -3

In [ ]:
# offline sanity checks first — if these fail, stop and fix before burning GPU time
!python -m src.data.negation --selftest
!python -m tests.test_data_pipeline | tail -3
!python -m tests.test_projection | tail -3
!python -m tests.test_nli_data | tail -3

## 2. HebNLI access

The dataset repo card marks it private, so the download needs a token. Store it as the
Colab secret `HF_TOKEN`; `src/data/hebnli.py` reads it from the environment.

In [ ]:
from google.colab import userdata
import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()

from huggingface_hub import whoami
print(whoami(token=os.environ['HF_TOKEN'])['name'])

## 3. Cache HebNLI

One download per split, parsed into our normalised shape. Everything after this reads
the local copy, so re-running the filters costs nothing.

`data/raw/` is gitignored — regenerate it, never commit it.

In [ ]:
!python -m src.data.hebnli --split train --out data/raw/hebnli_train.jsonl
!python -m src.data.hebnli --split val   --out data/raw/hebnli_val.jsonl
!python -m src.data.hebnli --split test  --out data/raw/hebnli_test.jsonl

## 4. Remove the contamination

Two filters, per split:

1. **promptID exclusion** — Itay's 689 held-out prompts. MultiNLI gives three
   hypotheses per premise sharing one promptID, so the whole prompt goes; dropping
   only the contradiction row would still train on our target as a premise next to
   our paraphrase.
2. **text-level audit** — the same Hebrew string can reach us under a *different*
   promptID, because HebNLI is machine-translated MultiNLI. An identifier filter
   cannot see that.

All three splits are filtered, not just train: if a probe sentence sat in val, the
validation accuracy would be inflated by the same contamination it is meant to check.

In [ ]:
!python -m src.nli.prepare_data --source data/raw/hebnli_train.jsonl --split train \
    --out data/raw/hebnli_train_clean.jsonl
!python -m src.nli.prepare_data --source data/raw/hebnli_val.jsonl --split val \
    --out data/raw/hebnli_val_clean.jsonl
!python -m src.nli.prepare_data --source data/raw/hebnli_test.jsonl --split test \
    --out data/raw/hebnli_test_clean.jsonl

In [ ]:
# The number nobody knows yet: how many rows the id filter passed but the text
# audit caught. Zero is a clean sentence for the methodology section; non-zero is
# a finding. Either way this manifest is what the report cites.
import json

for split in ('train', 'val', 'test'):
    m = json.load(open(f'results/nli_data_{split}.json', encoding='utf-8'))
    print(f"{split:6s} funnel={m['funnel']}  text_overlap={m['text_overlap']['rows_dropped']}")

## 5. Smoke run  ·  GPU from here

2000 rows, one epoch, a few minutes. This exercises tokenisation, the label map, the
training loop, checkpoint saving and the manifest — every part that can break — before
committing hours to the full run. `train_nli.py` has never been executed, so real bugs
surface here rather than in the filters.

Its numbers are marked `smoke_run` in the results table and must not be reported.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

!python -m src.nli.train_nli --base alephbert \
    --train data/raw/hebnli_train_clean.jsonl \
    --val data/raw/hebnli_val_clean.jsonl \
    --max-train 2000 --epochs 1

## 6. Full run

Two epochs over the whole clean train split — hours on a T4, and a free-tier session
ends when it ends. Two things follow, and both matter:

**The output must be on Drive.** `/content` is wiped when the runtime resets, so a
run that finishes into it leaves nothing behind. This project has already lost a run
to a Colab crash (commit `77cf572`).

**`--save-epochs` writes a resumable checkpoint after each epoch**, keeping the two
newest. Without it nothing exists on disk until training completes, so a disconnect
at 90% loses all of it. If the session does die, re-run the identical cell: it picks
up from the newest checkpoint automatically. `--fresh` is how you start over on
purpose.

Swap the base with one flag: `--base alephbertgimmel`. The checkpoint directory and
the manifest are both keyed on it, so two runs cannot overwrite each other, and
`results/nli_train.csv` gains a row per configuration for the comparison table.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CKPT = '/content/drive/MyDrive/hebrew-negation/checkpoints/alephbert-hebnli-clean'

In [ ]:
# re-run this exact cell after a disconnect - it resumes from the newest checkpoint
!python -m src.nli.train_nli --base alephbert \
    --train data/raw/hebnli_train_clean.jsonl \
    --val data/raw/hebnli_val_clean.jsonl \
    --save-epochs \
    --out "$CKPT"

## 7. Verify and take the results home

`check_nli_labels.py` prints raw probabilities for six obvious Hebrew pairs. Because we
wrote real names into `config.id2label`, the predictions should now agree with the
config instead of needing the hardcoded LABEL_0/1/2 guess the released checkpoint forced.

`results/*.csv` and `results/*.json` are small and belong in git so both of us see the
same numbers. Model weights do not — `.gitignore` blocks them, and they are on Drive.

In [ ]:
import pandas as pd
pd.read_csv('results/nli_train.csv')

In [ ]:
from google.colab import files
files.download('results/nli_train.csv')
files.download('results/nli_train_alephbert.json')
files.download('results/nli_data_train.json')
files.download('results/nli_data_val.json')
files.download('results/nli_data_test.json')

Commit them from your machine on `person-b`, then push.

**Still open after this notebook:** `nli_rerank.py` builds its input as one
`"premise [SEP] hypothesis [SEP]"` string, while this model is trained by passing the
pair to the tokenizer as two arguments. Inference has to reproduce what training used —
a mismatch degrades the model silently, with no error — so it needs an encoding switch
before it can load this checkpoint. The manifest records which mode was used.

Note that `alephbert-base` has `type_vocab_size=1`, so its segment ids are dropped;
`alephbertgimmel-base` has 2 and keeps them. `train_nli.py` reads this off the model
config rather than assuming, because feeding segment ids to AlephBERT is an IndexError
inside the embedding lookup, not a warning.